<a href="https://colab.research.google.com/github/zainabkhalid663/Flyrank-ML-Internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zainabkhalid663/Flyrank-ML-Internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row = one content item (page), scored once as of a decision date (end of March 2026).

Table: fact_content_daily_performance (HF warehouse), rolled up from daily to one row per
content_id for the month. I also join dim_content once for static fields (word_count,
content_type, days_since_last_update) — same content_id, not repeated per day.

Time window: month=2026-03 partition, 2026-03-01 to 2026-03-31. Mid-panel month, on purpose —
not the sealed test month (June 2026 _sample).

What I predict/rank: an opportunity score — how much a page deserves a refresh this month. Proxy,
not an observed outcome (same call as ML-03): built from position, demand, and trend, not from
actual refresh results, because no before/after refresh data exists yet.

What I exclude: fact_content_query_90d. Its 90-day window overlaps my decision month, so pulling
it in now risks mixing signal from after my decision point into a feature. Out of scope for this
notebook — revisit once I've aligned windows properly.

In [4]:
# setup: duckdb reads the HF parquet warehouse directly, no download
!pip -q install duckdb huggingface_hub scikit-learn

import os
import duckdb
from huggingface_hub import list_repo_files
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")
os.environ["HF_TOKEN"] = HF_TOKEN

REPO = "FlyRank/internship-warehouse"

all_files = list_repo_files(REPO, repo_type="dataset", token=HF_TOKEN)

fact_files = [f for f in all_files if "fact_content_daily_performance/" in f and "month=2026-03" in f]
dim_content_files = [f for f in all_files if "dim_content" in f and f.endswith(".parquet")]
dim_clients_files = [f for f in all_files if "dim_clients" in f and f.endswith(".parquet")]

print(f"{len(fact_files)} fact files for month=2026-03")
print(f"{len(dim_content_files)} dim_content files")
print(f"{len(dim_clients_files)} dim_clients files")
fact_files[:3]

1 fact files for month=2026-03
1 dim_content files
1 dim_clients files


['fact_content_daily_performance/month=2026-03/data_0.parquet']

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Feature (knowable before the decision moment, safe to use):
- avg_position — mean gsc_avg_position over March. Only uses days inside the window.
- total_impressions — sum of impressions over March. Same.
- ctr — clicks / impressions over March. Same.
- position_drift — second-half-of-March avg position minus first-half. Still fully inside the
  window, computed as of March 31, before the April decision.
- days_since_last_update — from dim_content, frozen as of the decision date. Content metadata,
  not a future peek.

Label / proxy: is_opportunity — rule: position in striking distance (5–20) AND impressions
≥ 100 AND position got worse in the second half of the month. Computed FROM the features above,
never itself a feature.

Context (grouping/joining only, never learned from): content_id, client_id.

Excluded:
- fact_content_query_90d — window overlap risk (see section 1), out of scope now.
- Any GA4 column where ga4_data_available is not TRUE — those are zero-filled placeholders for
  clients with no GA4 history yet, not real zero engagement.

In [9]:
from huggingface_hub import hf_hub_download

test_file = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    filename=fact_files[0],
    token=HF_TOKEN
)
print("downloaded fine:", test_file)

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  124MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

downloaded fine: /root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/fact_content_daily_performance/month=2026-03/data_0.parquet


In [10]:
con = duckdb.connect()
con.sql("INSTALL httpfs; LOAD httpfs;")
con.sql(f"CREATE OR REPLACE SECRET hf_secret (TYPE huggingface, TOKEN '{HF_TOKEN}');")

fact_paths_sql = ", ".join(f"'hf://datasets/{REPO}/{f}'" for f in fact_files)
con.sql(f"CREATE OR REPLACE VIEW march AS SELECT * FROM read_parquet([{fact_paths_sql}])")

con.sql("SELECT * FROM march LIMIT 3").df()

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,<NA>,20,0,67,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,<NA>,1,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,<NA>,125,1,616,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Three queries: grain, counts + date span, availability (IS TRUE). Then the five-feature frame,
then the trap.

In [12]:
con.sql("DESCRIBE march").df()

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


In [13]:
con.sql("""
SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS c
FROM march
GROUP BY report_date, client_hash_id, content_hash_id
HAVING c > 1
LIMIT 5
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,c


In [14]:
con.sql("""
SELECT COUNT(*) AS n_rows,
       COUNT(DISTINCT content_hash_id) AS n_content,
       COUNT(DISTINCT client_hash_id) AS n_clients,
       MIN(report_date) AS min_date,
       MAX(report_date) AS max_date
FROM march
""").df()

,n_rows,n_content,n_clients,min_date,max_date
0,9841378,331437,55,2026-03-01,2026-03-31


In [15]:
con.sql("""
SELECT COUNT(*) AS total_rows,
       SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS ga4_available_rows,
       ROUND(100.0 * SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_available
FROM march
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,ga4_available_rows,pct_available
0,9841378,413966.0,4.2


In [16]:
con.sql(f"""
CREATE OR REPLACE VIEW dim_content AS
SELECT * FROM read_parquet([{', '.join(repr('hf://datasets/' + REPO + '/' + f) for f in dim_content_files)}])
""")

con.sql("DESCRIBE dim_content").df()

,column_name,column_type,null,key,default,extra
0,client_hash_id,VARCHAR,YES,None,None,None
1,content_hash_id,VARCHAR,YES,None,None,None
2,keyword_hash_id,VARCHAR,YES,None,None,None
3,url_hash_id,VARCHAR,YES,None,None,None
4,keyword_char_count,BIGINT,YES,None,None,None
5,keyword_token_count,BIGINT,YES,None,None,None
6,url_char_count,BIGINT,YES,None,None,None
7,content_created_date,DATE,YES,None,None,None
8,content_updated_date,DATE,YES,None,None,None
9,content_type,VARCHAR,YES,None,None,None


In [17]:
con.sql("""
SELECT COUNT(*) AS n_rows, COUNT(DISTINCT content_hash_id) AS n_distinct_content
FROM dim_content
""").df()

,n_rows,n_distinct_content
0,519606,519606


In [24]:
sql_parts = [
    "WITH agg AS (",
    "  SELECT",
    "    content_hash_id,",
    "    ANY_VALUE(client_hash_id) AS client_hash_id,",
    "    AVG(gsc_avg_position) AS avg_position,",
    "    SUM(gsc_impressions) AS total_impressions,",
    "    SUM(gsc_clicks) AS total_clicks,",
    "    AVG(CASE WHEN report_date <= DATE '2026-03-15' THEN gsc_avg_position END) AS first_half_position,",
    "    AVG(CASE WHEN report_date >  DATE '2026-03-15' THEN gsc_avg_position END) AS second_half_position",
    "  FROM march",
    "  GROUP BY content_hash_id",
    ")",
    "SELECT",
    "  a.content_hash_id,",
    "  a.avg_position,",
    "  a.total_impressions,",
    "  ROUND(100.0 * a.total_clicks / NULLIF(a.total_impressions, 0), 2) AS ctr,",
    "  (a.second_half_position - a.first_half_position) AS position_drift,",
    "  GREATEST(",
    "    CASE",
    "      WHEN d.content_updated_date <= DATE '2026-03-31'",
    "      THEN DATE_DIFF('day', d.content_updated_date, DATE '2026-03-31')",
    "      ELSE DATE_DIFF('day', d.content_created_date, DATE '2026-03-31')",
    "    END, 0",
    "  ) AS days_since_last_update",
    "FROM agg a",
    "JOIN dim_content d USING (content_hash_id)",
]
sql = "\n".join(sql_parts)

feature_frame = con.sql(sql).df()
print("before dropping no-visibility rows:",

before dropping no-visibility rows: (331437, 6)
after dropping no-visibility rows: (176738, 6)


,content_hash_id,avg_position,total_impressions,ctr,position_drift,days_since_last_update
0,content_7a105f548d9c6916,7.209549,6523.0,0.11,1.709337,396
1,content_a3ea9792f793ec72,2.987198,453.0,0.00,-1.781831,396
2,content_36c36abc7650d7af,6.724039,5630.0,0.11,0.484963,396
3,content_a7da352b73b02668,7.244844,4944.0,0.26,-0.029095,396
4,content_f39be42b42a4e8f6,14.432540,42.0,0.00,9.506944,396


In [25]:
feature_frame["days_since_last_update"].describe()

,days_since_last_update
count,176738.000000
mean,161.432878
std,131.319127
min,0.000000
25%,34.000000
50%,113.000000
75%,248.000000
max,494.000000


In [27]:
feature_frame["is_opportunity"] = (
    feature_frame["avg_position"].between(5, 20)
    & (feature_frame["total_impressions"] >= 100)
    & (feature_frame["position_drift"] > 0)
).astype(int)

print(feature_frame["is_opportunity"].sum(), "/", len(feature_frame), "flagged")
print(f"{feature_frame['is_opportunity'].mean():.1%}")

28096 / 176738 flagged
15.9%


In [28]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

feature_cols = ["avg_position", "total_impressions", "ctr", "position_drift", "days_since_last_update"]
X = feature_frame[feature_cols].fillna(0)
y = feature_frame["is_opportunity"]

honest_auc = cross_val_score(LogisticRegression(max_iter=1000), X, y, cv=5, scoring="roc_auc").mean()
print("honest AUC (5 real features, no leak):", round(honest_auc, 3))

honest AUC (5 real features, no leak): 0.693


In [29]:
X_leaked = X.copy()
X_leaked["opportunity_hint"] = y  # the label itself, snuck in as a feature

leaked_auc = cross_val_score(LogisticRegression(max_iter=1000), X_leaked, y, cv=5, scoring="roc_auc").mean()
print("leaked AUC (label snuck in as a feature):", round(leaked_auc, 3))
print(f"jump: +{leaked_auc - honest_auc:.3f} toward perfect, from one column")

del X_leaked
print("keeping the honest number:", round(honest_auc, 3))

leaked AUC (label snuck in as a feature): 1.0
jump: +0.307 toward perfect, from one column
keeping the honest number: 0.693


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

One month is a short window for "trend." A first-half-vs-second-half split inside March is noisy
— two weeks of position data can swing from normal ranking volatility, not a real refresh signal.
A trustworthy trend line needs 60–90 days, and 17 of my 55 clients (31%) only have GSC history
starting after Jan 1, 2026 — under that runway by March. Also worth flagging: only 4.2% of March
rows have real GA4 data (from query 3) — this slice is mostly GSC-only, so engagement-based
signals aren't reliable here yet.

In [30]:
sql_parts = [
    "CREATE OR REPLACE VIEW dim_clients AS",
    "SELECT * FROM read_parquet([" +
    ", ".join(repr("hf://datasets/" + REPO + "/" + f) for f in dim_clients_files) +
    "])",
]
con.sql("\n".join(sql_parts))

con.sql("DESCRIBE dim_clients").df()

,column_name,column_type,null,key,default,extra
0,client_hash_id,VARCHAR,YES,None,None,None
1,is_active,BOOLEAN,YES,None,None,None
2,has_gsc_access,BOOLEAN,YES,None,None,None
3,has_ga4_access,BOOLEAN,YES,None,None,None
4,access_profile,VARCHAR,YES,None,None,None
5,client_created_date,DATE,YES,None,None,None
6,client_updated_date,DATE,YES,None,None,None
7,gsc_data_start,DATE,YES,None,None,None
8,ga4_data_start,DATE,YES,None,None,None


In [31]:
sql_parts = [
    "SELECT COUNT(*) AS n_clients,",
    "       SUM(CASE WHEN gsc_data_start > DATE '2026-01-01' THEN 1 ELSE 0 END) AS short_history_clients",
    "FROM dim_clients",
    "WHERE client_hash_id IN (SELECT DISTINCT client_hash_id FROM march)",
]
con.sql("\n".join(sql_parts)).df()

,n_clients,short_history_clients
0,55,17.0


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.